In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

%matplotlib inline

In [ ]:
df=pd.read_csv('clean_data.csv')

In [ ]:
df

In [ ]:
X=df.drop('Discount Price',axis=1)
y=df['Discount Price']

In [ ]:
df

In [ ]:
X

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lm=LinearRegression()

In [ ]:
lm.fit(X_train,y_train)

In [ ]:
y_pred=lm.predict(X_test)

In [ ]:
y_pred

In [ ]:
y_test

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
res = r2_score(y_test,y_pred)

In [ ]:
print(res)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
dt_model=DecisionTreeRegressor()

In [ ]:
dt_model.fit(X_train,y_train)

In [ ]:
y_pred=dt_model.predict(X_test)

In [ ]:
res=r2_score(y_test,y_pred)

In [ ]:
print(res)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor()
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)
res=r2_score(y_test,y_pred)
print(res)

In [ ]:
pip install xgboost

In [ ]:
import xgboost as xgb

In [ ]:
model=xgb.XGBRegressor()

In [ ]:
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
res=r2_score(y_pred,y_test)
print(res)

In [ ]:
from sklearn.model_selection import cross_val_score
score_decision=cross_val_score(dt_model,X,y,cv=5,scoring='r2')
score_random=cross_val_score(rf,X,y,cv=5)
score_xgb=cross_val_score(model,X,y,cv=5)
print(score_decision)
print(score_random)
print(score_xgb)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
para_Grid={
    'n_estimators':[100,200,300],
    'max_depth':[3,4,5],
    'learning_rate':[0.01,0.1,0.2]
}

Good—this is a small piece of code, but it’s doing something **very important**.

Let’s break both lines clearly 👇

---

# 🔹 Line 1

```python
best_param = grid_search.best_params_
```

## 🧠 What it does

👉 It **extracts the best hyperparameters** found by GridSearchCV

After trying all combinations + cross-validation, GridSearch stores:

[
\theta^* = \arg\max_{\theta} \text{CV Score}(\theta)
]

👉 `best_params_` = that optimal (\theta^*)

---

## 📦 Example output

```python
best_param = {
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.1
}
```

👉 These are the **best settings for your model**

---

# 🔹 Line 2

```python
best_model = xgb.XGBRegressor(**best_param)
```

## 🧠 What it does

👉 It creates a **new model using those best hyperparameters**

---

## ⚙️ What does `**best_param` mean?

This is Python unpacking:

```python
XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1
)
```

👉 Same thing, just shorter

---

# 🔥 Why do we do this?

Because:

* GridSearchCV only **finds best parameters**
* Now we want to:

  * Build a clean model
  * Train it on full data

---

# ⚠️ Important (many people miss this)

You actually **don’t need to do this manually**

You can directly use:

```python
best_model = grid_search.best_estimator_
```

👉 This already:

* Has best parameters
* Is already trained

---

# 🧠 Full flow

```python
grid_search.fit(X_train, y_train)

best_param = grid_search.best_params_

best_model = xgb.XGBRegressor(**best_param)

best_model.fit(X_train, y_train)
```

---

# 🧠 What’s happening conceptually

1. GridSearch → finds best hyperparameters
2. You → rebuild model with those hyperparameters
3. Train → final model

---

# 🧠 One-line memory

> `best_params_` = best settings
> `XGBRegressor(**best_param)` = model built using those settings

---

If you want next:

👉 I can show **what happens internally when you pass `max_depth`, `learning_rate` into XGBoost mathematically** (this connects everything you learned so far)


In [ ]:
grid_search=GridSearchCV(estimator=model,param_grid=para_Grid,cv=5,scoring='r2')
grid_search.fit(X_train,y_train)
best_param=grid_search.best_params_
best_model=xgb.XGBRegressor(**best_param)
best_model.fit(X_train,y_train)
y_pred=best_model.predict(X_test)
r2=r2_score(y_test,y_pred)
print(r2)

In [ ]:
plt.scatter(y_test,y_pred,alpha=0.5)
plt.plot([min(y_test),max(y_test)],[min(y_test),max(y_test)],'r',linestyle='-')
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title('Actual vs Predicted Values')
plt.show()

In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [ ]:
with open('model.pkl','rb') as f:
    xgb_model=pickle.load(f)

In [ ]:
y_pred=xgb_model.predict(X_test)
print(r2_score(y_test,y_pred))